The code below iterated over all of the median areas I discovered and buffered them, joining them into a single median area for each county. Aside from needing to update column names, this code will likely function correctly once I correct earlier code \(which only grabbed the first daily median per county\) but I should doublecheck everything is accurate.


In [0]:
#Purpose of Cell Block: Iterate over counties that have not yet been processed to generate and checkpoint dissolved 1-km buffer geometries around unique median-cell locations for each county, using a restartable workflow that detects completed outputs, skips finished counties, writes results safely to disk, and manages memory to support long-running geospatial processing.

sf_use_s2(FALSE)

checkpoint_dir <- "D_processed_data/intermediate_scratch/checkpoints_county_buffers"
dir.create(checkpoint_dir, recursive = TRUE, showWarnings = FALSE)

#BEGIN FUNCTION: CHECKPOINT FILENAME AND PATH
checkpoint_path <- function(county_i) {
  file.path(checkpoint_dir, sprintf("buffers_1km_dissolved_county_%05d.gpkg", county_i))
}
#END FUNCTION: CHECKPOINT FILENAME AND PATH

counties_all <- sort(unique(county_multi_cell$county_row)) #Identify the total number of counties

existing <- list.files(checkpoint_dir, pattern = "^buffers_1km_dissolved_county_[0-9]{5}\\.gpkg$", full.names = FALSE) #Check the files already in the checkpoint directory
completed <- as.integer(str_extract(existing, "[0-9]{5}")) #grab the number of complted files

counties_todo <- setdiff(counties_all, completed) #find difference between the total number of counties and the number completed

#Provide information on status
length(counties_all)
length(counties_todo)
head(counties_todo)


counter <- length(counties_all) - length(counties_todo) #status update initiation


for (i in counties_todo) {

  counter <- counter + 1 #status bar
  cat(sprintf("\nProcessing county %d of %d (county_row=%d)\n", counter, length(counties_all), i))
  flush.console()

  pts_df <- county_multi_cell %>% #grab points for county[i], dropping any duplicated xy values
    filter(county_row == i) %>%
    distinct(x, y)

  if (nrow(pts_df) == 0) next   # If a county has no points, skip to next i 

  # Points -> buffers (1km) -> dissolve overlaps for county[i]
  buffers_dissolved <- pts_df %>%
    st_as_sf(coords = c("x", "y"), crs = 4326) %>% # EPSG:4326 Geodetic coordinate system 
    st_transform(3857) %>%
    st_buffer(1000) %>% #buffer to 1km
    st_union() #union to create one feature for the entire county

  out_sf <- st_sf( #prepare file to save as checkpoint
    county_row = i,
    geometry = buffers_dissolved,
    crs = 3857
  )

  out_file <- checkpoint_path(i)
  tmp_file <- sub("\\.gpkg$", ".tmp.gpkg", out_file) #prevent possibility ill create a corrupted file if cocalc times out while saving

  # Write temp then rename
  st_write(out_sf, tmp_file, delete_dsn = TRUE, quiet = TRUE)
  file.rename(tmp_file, out_file)
    
  # Free RAM 
  rm(pts_df, buffers_dissolved, out_sf)
  gc()
}

#Conclusion: 3108 .gpkg were exported (1 file of 1km buffered and dissolved points for each contiguous US county)

In [0]:


setwd("/home/user/capstone/A_data")

# Load counties once
contiguous_counties <- vect(
  "D_processed_data/intermediate_scratch/contiguous_counties_geom.shp"
)
counties_sf <- st_as_sf(contiguous_counties)

# Locate buffer files
checkpoint_dir <- "D_processed_data/intermediate_scratch/checkpoints_county_buffers"

buffer_files <- list.files(
  checkpoint_dir,
  pattern = "^buffers_1km_dissolved_county_[0-9]{5}\\.gpkg$",
  full.names = TRUE
)

# Randomly select five files
set.seed(42)
sample_files <- sample(buffer_files, 5)

for (f in sample_files) {

  buffers <- st_read(f, quiet = TRUE)

  # Reproject counties to buffer CRS
  counties_proj <- st_transform(counties_sf, st_crs(buffers))

  # --- ZOOM EXTENT FROM BUFFER ---
  bb <- st_bbox(buffers)

  print(
    ggplot() +
      geom_sf(
        data = counties_proj,
        fill = NA,
        color = "grey80",
        linewidth = 0.2
      ) +
      geom_sf(
        data = buffers,
        fill = "steelblue",
        alpha = 0.5,
        color = NA
      ) +
      coord_sf(
        xlim = c(bb["xmin"], bb["xmax"]),
        ylim = c(bb["ymin"], bb["ymax"]),
        expand = FALSE
      ) +
      labs(title = basename(f)) +
      theme_minimal()
  )
}
